# 02 - Preparação dos dados
**Projeto:** PI 4CCOMP 2026/2, CTI Global, Grupo 6
**Fase CRISP-DM:** Preparação dos Dados

Este notebook executa e explica, etapa por etapa, a preparação que gera a **base analítica única**
do projeto. É essa base que a Análise Inferencial, Contabilidade e Finanças e o dashboard vão usar,
para que todas as entregas partam dos mesmos números.

| Tarefa do CRISP-DM | Seção |
|---|---|
| Selecionar os dados | 2 |
| Limpar e uniformizar | 3 e 4 |
| Integrar os dados | 5 |
| Derivar os dados | 6 |
| Formatar os dados | 5 e 8 |

As tarefas de coletar, descrever, explorar e verificar a qualidade estão no notebook `01_perfil.ipynb`.

A lógica fica no script `src/prepara.py`. Este notebook chama as funções dele e mostra o resultado de
cada uma, sem reescrever as regras em outro lugar.

## 0. Como executar

**Na máquina local:** abra o notebook com o kernel do ambiente `.venv` e execute as células em ordem.
A base precisa estar em `src/data/raw/Demonstrativo Fecap v3.csv`.

**No Google Colab:** a célula abaixo clona o repositório e copia a base a partir do Google Drive.
Ajuste `CSV_NO_DRIVE` para o caminho onde a base está no seu Drive. Fora do Colab, ela não faz nada.

In [1]:
import os
import shutil
import subprocess
import sys

EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    CSV_NO_DRIVE = "/content/drive/MyDrive/Base CTI/Demonstrativo Fecap v3.csv"
    from google.colab import drive

    drive.mount("/content/drive")
    if not os.path.exists("/content/Projeto6"):
        subprocess.run(["git", "clone", "-q", "https://github.com/2026-2-NCC4/Projeto6.git",
                        "/content/Projeto6"], check=True)
    os.chdir("/content/Projeto6")
    shutil.copy(CSV_NO_DRIVE, "src/data/raw/Demonstrativo Fecap v3.csv")
    print("Colab preparado: repositório clonado e base copiada do Drive")
else:
    print("Execução local: usando o repositório e a base já presentes na máquina")

Execução local: usando o repositório e a base já presentes na máquina


In [2]:
from pathlib import Path

import pandas as pd

# Sobe a partir da pasta atual até encontrar a raiz do repositório
raiz = Path.cwd()
while not (raiz / "src" / "prepara.py").exists() and raiz != raiz.parent:
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))
import prepara  # noqa: E402

p = prepara.caminhos(raiz)
print("raiz do projeto:", raiz)
print("base encontrada:", p["bruto"].exists())

raiz do projeto: C:\Users\Gusta\OneDrive\Documentos\GitHub\Projeto6
base encontrada: True


## 1. Coletar e registrar a entrada

A leitura segue o contrato definido no notebook 01: separador `;`, sem cabeçalho, codificação latin-1 e
números no formato brasileiro. Cada linha recebe `linha_origem`, o número dela no arquivo original,
para que qualquer valor da base preparada possa ser rastreado até a fonte.

Antes de ler, calculamos o **SHA-256** do arquivo, uma "impressão digital". Se ela for igual no fim da
preparação, fica provado que a base da CTI não foi alterada.

In [3]:
hash_antes = prepara.hash_arquivo(p["bruto"])
bruto = prepara.coletar(p["bruto"])

print("linhas lidas     :", len(bruto))
print("SHA-256 (início) :", hash_antes[:20], "...")
bruto.head()

linhas lidas     : 1002000
SHA-256 (início) : a93fb4555eab2eee7a43 ...


,linha_origem,ano,cenario,conta,valor
0,1,Ano 1,Total Cen_00001,NaN,2.513881e-02
1,2,Ano 1,Total Cen_00001,BAL - Total do Ativo,1.063113e+10
2,3,Ano 1,Total Cen_00001,BAL - Ativo Circulante,1.795721e+09
3,4,Ano 1,Total Cen_00001,BAL - Disponível,1.205265e+09
4,5,Ano 1,Total Cen_00001,BAL - Contas a Receber - SWAP,9.568042e+07


## 2. Selecionar os dados

O arquivo tem dois tipos de linha: **contas financeiras** e **linhas sem nome de conta**. A verificação
V5 do notebook 01 mostrou que a primeira linha sem nome de cada grupo é o resíduo de conferência do
balanço, e que as outras duas são sempre zero. Nenhuma delas é conta financeira.

**Decisão:** as contas financeiras seguem para a base analítica. As linhas sem nome **não são apagadas**:
vão para um arquivo de auditoria, e a primeira delas volta à base como a coluna `conferencia_modelo`.

In [4]:
nomeadas, sem_conta = prepara.selecionar(bruto)

print("contas financeiras:", len(nomeadas))
print("linhas sem conta  :", len(sem_conta))
print("soma              :", len(nomeadas) + len(sem_conta), "| lidas:", len(bruto))
print()
print("linhas sem conta, por posição no grupo:")
print(sem_conta.groupby("posicao")["valor"].agg(["count", "min", "max"]).to_string())

contas financeiras: 958800
linhas sem conta  : 43200
soma              : 1002000 | lidas: 1002000

linhas sem conta, por posição no grupo:
         count      min       max
posicao                          
1        14400 -0.02511  0.025153
2        14400  0.00000  0.000000
3        14400  0.00000  0.000000


## 3. Limpar e uniformizar

Parte das contas foi gravada com dois espaços antes do hífen (`"BAL  - Capital Social"`), e parte com um
(`"BAL - Capital Social"`). Isso não cria contas duplicadas, porque cada conta usa sempre a mesma grafia.
Mas **quebra o agrupamento por bloco**: ao separar o prefixo, o espaço extra gera um bloco `"BAL "`
diferente de `"BAL"`.

**Decisão:** criar a coluna `conta` com os espaços padronizados e **manter o nome original** em
`conta_original`. Também são criados `bloco` (BAL, DRE ou FLU), `ano_n` (1 a 12) e `ano_calendario`
(2027 a 2038, pela convenção da CTI).

Contas de nome parecido, como `Emprést` e `Empréstimos`, **não são unidas**: estão em grupos diferentes
do balanço e têm sinais diferentes.

In [5]:
bloco_antes = nomeadas["conta"].str.split(" - ").str[0].value_counts()
nomes_antes = nomeadas["conta"].nunique()

nomeadas = prepara.uniformizar(nomeadas)

afetadas = nomeadas.loc[nomeadas["conta_original"] != nomeadas["conta"], "conta"].nunique()
print("nomes de conta distintos: antes", nomes_antes, "| depois", nomeadas["conta"].nunique())
print("contas com grafia corrigida:", afetadas)
print()
print("prefixo de bloco ANTES:")
print(bloco_antes.to_string())
print()
print("prefixo de bloco DEPOIS:")
print(nomeadas["bloco"].value_counts().to_string())

nomes de conta distintos: antes 69 | depois 69
contas com grafia corrigida: 9

prefixo de bloco ANTES:
conta
BAL     468000
DRE     201600
FLU     172800
BAL     116400

prefixo de bloco DEPOIS:
bloco
BAL    584400
DRE    201600
FLU    172800


## 4. Validar a chave antes de integrar

Na próxima etapa, cada conta vira uma coluna. Se uma combinação (cenário, ano, conta) aparecesse duas
vezes, uma transformação com soma juntaria os dois valores **sem avisar**, e o número errado seguiria
para todas as análises.

**Decisão:** validar a chave antes e interromper a preparação se houver repetição. A segunda parte da
célula prova que a trava funciona: inserimos uma duplicata de propósito numa cópia e a validação recusa.

In [6]:
prepara.validar_chave(nomeadas)
print("Chave válida: nenhuma combinação (cenário, ano, conta) repetida.")
print()

teste = pd.concat([nomeadas, nomeadas.head(1)])  # cópia com uma linha duplicada de propósito
try:
    prepara.validar_chave(teste)
except ValueError as erro:
    print("Teste com duplicata inserida -> a validação recusou:")
    print(" ", str(erro).splitlines()[0])

Chave válida: nenhuma combinação (cenário, ano, conta) repetida.



Teste com duplicata inserida -> a validação recusou:
  2 linhas com chave repetida:


## 5. Integrar e formatar

Os três demonstrativos (BAL, DRE e FLU) estão em linhas separadas. A integração os reúne numa
**linha por cenário e ano**, com cada conta numa coluna: 1.200 cenários x 12 anos = 14.400 linhas.

**Contas ausentes continuam vazias**, não viram zero. O Ano 12 é o encerramento da concessão e não traz
27 contas de detalhe; preencher com zero afirmaria um valor que a CTI não informou.

**Fonte externa:** nenhuma foi integrada nesta entrega. O IPCA foi considerado para converter valores
nominais, mas a CTI não informou se a base já está ajustada; integrar sem essa resposta poderia
distorcer os números.

In [7]:
base, contas = prepara.formatar(nomeadas)
print("base analítica:", base.shape[0], "linhas x", base.shape[1], "colunas")
print("contas como colunas:", len(contas))
print()

ausentes = base.groupby("ano_calendario")[contas].apply(lambda g: int(g.isna().all().sum()))
print("contas ausentes por ano:")
print(ausentes.to_string())

base[["cenario", "ano_n", "ano_calendario", "DRE - Receita", "DRE - EBITDA", "FLU - Saldo Final"]].head()

base analítica: 14400 linhas x 72 colunas
contas como colunas: 69

contas ausentes por ano:
ano_calendario
2027     1
2028     1
2029     0
2030     0
2031     0
2032     0
2033     0
2034     0
2035     0
2036     0
2037     0
2038    27


,cenario,ano_n,ano_calendario,DRE - Receita,DRE - EBITDA,FLU - Saldo Final
0,Total Cen_00001,1,2027,6.722729e+09,5.188381e+09,1.205265e+09
1,Total Cen_00001,2,2028,6.114937e+09,4.630952e+09,1.022427e+09
2,Total Cen_00001,3,2029,6.251283e+09,4.883290e+09,1.748421e+09
3,Total Cen_00001,4,2030,6.506532e+09,5.148066e+09,1.117390e+09
4,Total Cen_00001,5,2031,6.844711e+09,5.453240e+09,3.073864e+08


## 6. Derivar

Novas colunas criadas, **sem alterar nenhuma conta original**:

| Coluna | Para que serve |
|---|---|
| `encerramento` | Marca o Ano 12, para as análises de tendência poderem separá-lo |
| `residuo_balanco` | Ativo + Passivo; deve ser próximo de zero |
| `conferencia_modelo` | A primeira linha sem conta, trazida de volta para comparação |
| `margem_ebitda`, `margem_operacional`, `margem_liquida` | Resultado dividido pela receita |
| `grupo_repeticao`, `cenario_repetido`, `copia_excedente` | Identificam os cenários com trajetória idêntica |

**Cenários repetidos:** a comparação usa os valores completos, sem arredondar, e cada grupo é conferido
valor a valor. **Todos os 1.200 cenários ficam na base.** Quem quiser analisar sem as cópias filtra
`copia_excedente == False`. Essa escolha deve ser declarada em cada análise.

O EBITDA é usado **como a CTI forneceu**. Ele não reconcilia com Resultado Operacional menos
Depreciação e Amortização, e a memória de cálculo foi pedida à empresa.

In [8]:
base, marcas = prepara.derivar(base, sem_conta, contas)

print("cenários repetidos :", int(marcas["cenario_repetido"].sum()))
print("grupos             :", marcas["grupo_repeticao"].nunique())
print("cópias excedentes  :", int(marcas["copia_excedente"].sum()))
print("cenários sem cópias:", int((~marcas["copia_excedente"]).sum()))
print()
print("mediana das margens por ano, entre os cenários:")
margens = ["margem_ebitda", "margem_operacional", "margem_liquida"]
print(base.groupby("ano_calendario")[margens].median().round(3).to_string())

cenários repetidos : 94
grupos             : 1
cópias excedentes  : 93
cenários sem cópias: 1107

mediana das margens por ano, entre os cenários:
                margem_ebitda  margem_operacional  margem_liquida
ano_calendario                                                   
2027                    0.771               0.636           0.364
2028                    0.758               0.581           0.326
2029                    0.783               0.596           0.340
2030                    0.794               0.602           0.357
2031                    0.800               0.631           0.387
2032                    0.806               0.643           0.408
2033                    0.811               0.640           0.411
2034                    0.815               0.646           0.417
2035                    0.818               0.652           0.427
2036                    0.823               0.679           0.453
2037                    0.824               0.712           0.

## 7. Verificar a base preparada

As identidades contábeis são testadas de novo, agora na base preparada. Cada teste informa quantos
grupos foram comparáveis: o teste do passivo compara 13.200 grupos, e não 14.400, porque o Ano 12 não
tem Patrimônio Líquido nem Exigível a Longo Prazo.

In [9]:
relatorio = prepara.verificar(bruto, nomeadas, sem_conta, base, marcas)
pd.DataFrame(relatorio["testes"])[["teste", "comparaveis", "aprovados", "passou"]]

,teste,comparaveis,aprovados,passou
0,Total do Ativo + Total do Passivo = 0,14400,14400,True
1,Ativo = Circulante + Realizável LP + Permanente,14400,14400,True
2,Passivo = Circulante + Exigível LP + Patrimôni...,13200,13200,True
3,Saldo Final = Saldo Inicial + Geração de Caixa,14400,14400,True
4,Conferência do modelo = resíduo do balanço,14400,14400,True


## 8. Gerar os arquivos oficiais

A execução completa do script grava as saídas e confere a impressão digital da entrada no fim.
É o mesmo resultado de rodar `python src/prepara.py` no terminal.

| Arquivo | Onde | No GitHub? |
|---|---|---|
| `base_analitica.parquet` | `src/data/processed/` | Não: contém valores da CTI |
| `base_longa.parquet` | `src/data/processed/` | Não |
| `auditoria_sem_conta.parquet` | `src/data/processed/` | Não |
| `relatorio_qualidade.json` | `src/data/processed/` | Não |
| `dicionario_dados.md` | pasta do PI em `documentos/Entrega 1/` | Sim: só nomes e descrições |

In [10]:
base_oficial, relatorio_oficial = prepara.main(raiz)

print()
print("igual à base construída passo a passo:",
      base_oficial.equals(base))
print("entrada inalterada ao final:", prepara.hash_arquivo(p["bruto"]) == hash_antes)

Base analítica: 14400 linhas (1200 cenários x 12 anos), 69 contas
Cenários repetidos: 94 em 1 grupo(s); 93 cópias excedentes
[OK  ] Total do Ativo + Total do Passivo = 0: 14400/14400
[OK  ] Ativo = Circulante + Realizável LP + Permanente: 14400/14400
[OK  ] Passivo = Circulante + Exigível LP + Patrimônio Líquido: 13200/13200
[OK  ] Saldo Final = Saldo Inicial + Geração de Caixa: 14400/14400
[OK  ] Conferência do modelo = resíduo do balanço: 14400/14400
Arquivo bruto inalterado (SHA-256 igual antes e depois).

igual à base construída passo a passo: True
entrada inalterada ao final: True


## 9. Resumo da preparação

| Tarefa CRISP-DM | O que foi feito | Evidência |
|---|---|---|
| Selecionar | Contas financeiras separadas das linhas sem nome, que foram preservadas em auditoria | Seção 2 |
| Limpar | Chave validada; nenhuma duplicata. Trava testada com duplicata proposital | Seção 4 |
| Uniformizar | Espaços padronizados nos nomes, nome original preservado, blocos corrigidos de 4 para 3 | Seção 3 |
| Integrar | BAL, DRE e FLU reunidos numa linha por cenário e ano. Nenhuma fonte externa, com justificativa | Seção 5 |
| Derivar | Marcação do encerramento, conferência do balanço, três margens e marcação das repetições | Seção 6 |
| Formatar | Base em Parquet com 14.400 linhas e dicionário de dados em Markdown | Seções 5 e 8 |

### Decisões registradas

1. Linhas sem nome de conta não entram como contas, mas são preservadas e a primeira volta como conferência.
2. Contas ausentes ficam vazias, nunca zero.
3. O Ano 12 permanece na base, marcado como encerramento.
4. Os 1.200 cenários permanecem; as 93 cópias são marcadas, não removidas.
5. Contas de nome parecido não são unidas.
6. O EBITDA é usado como fornecido pela CTI, até a resposta sobre a memória de cálculo.
7. Nenhuma fonte externa foi integrada nesta entrega.
8. O arquivo original da CTI não é alterado, o que é provado pela impressão digital igual no início e no fim.

### Próximos passos

A base analítica está pronta para a **Análise Inferencial** (medidas descritivas, histogramas e boxplots)
e para **Contabilidade e Finanças** (demonstrativos e índices).